Jupyter's kernel already runs an event loop internally — `asyncio.run()` tries to create and run a new event loop, which is illegal while one is already running.

> The fix: `await main()` directly (Jupyter lets you await at the top level of a cell).

---

### Race Condition in Asyncio

In [1]:
import asyncio

counter = 0


async def increment():
    global counter
    value = counter  # read
    await asyncio.sleep(0)  # yields control, another coroutine can run
    counter = value + 1  # write


async def main():
    tasks = [increment() for _ in range(100)]
    await asyncio.gather(*tasks)
    print(counter)


# asyncio.run() fails in Jupyter because the kernel already runs an event loop.
# In Jupyter, top-level `await` is supported directly.
await main()


1


**What's happening:**

```python
async def increment():
    value = counter      # read
    await asyncio.sleep(0)  # ← yields control here
    counter = value + 1  # write
```

`asyncio.sleep(0)` yields back to the event loop without sleeping. All 100 tasks read `counter` (= 0) before any of them write back. Then all 100 write `0 + 1 = 1`. Result: `1` instead of `100`.

This is a **race condition in asyncio** — which seems to contradict "asyncio is safe from race conditions." The nuance:

- asyncio is safe _as long as you don't `await` in the middle of a read-modify-write_.
- `await` is a **yield point** — another coroutine can run between `value = counter` and `counter = value + 1`.
- The rule: any shared state mutation that spans an `await` is a critical section and needs an `asyncio.Lock`.

The fix in real code would be wrapping the read-modify-write with `async with asyncio.Lock()`. But the example is showing you the trap, not fixing it — keep it as-is to have the bad case documented.

---

### Fixed version — `asyncio.Lock`

Wrap the entire read-modify-write in a lock so no other coroutine can enter that critical section while one is between the `read` and the `write`.

In [2]:
import asyncio

counter = 0
lock = asyncio.Lock()


async def increment():
    global counter
    async with lock:          # only one coroutine inside here at a time
        value = counter       # read
        await asyncio.sleep(0)  # yield — but no other coroutine can enter the lock
        counter = value + 1   # write


async def main():
    tasks = [increment() for _ in range(100)]
    await asyncio.gather(*tasks)
    print(counter)  # always 100


await main()


100


The key contrast to understand:

- **Broken**: `await` inside a naked read-modify-write → event loop switches to another coroutine mid-operation → stale reads pile up → result: `1`
- **Fixed**: `async with lock` holds the lock across the `await` → any other coroutine that reaches `async with lock` suspends _at the lock entry_, not inside the critical section → result: always `100`

One thing worth noting: `asyncio.Lock` is **not** interchangeable with `threading.Lock`. `asyncio.Lock` only works within the event loop (it suspends a coroutine, not a thread). If you accidentally used `threading.Lock` here it would _work_ by coincidence (single thread, so never actually contended), but it's semantically wrong and would deadlock if used in a thread pool context.

**All reading the same stale value(captured snapshot) before any write has happened**.

The precise sequence in the broken version:

```shell
coroutine 1:  read counter=0  →  await (yields)
coroutine 2:  read counter=0  →  await (yields)
coroutine 3:  read counter=0  →  await (yields)
... (all 100 read 0, all yield)
coroutine 1:  write counter = 0+1 = 1
coroutine 2:  write counter = 0+1 = 1  ← overwrites, not 2
coroutine 3:  write counter = 0+1 = 1  ← overwrites, not 3
... (all 100 write 1)
result: 1
```

And the fixed version:

```shell
coroutine 1:  acquire lock  →  read counter=0  →  await (yields, but holds lock)
coroutine 2:  tries to acquire lock  →  suspended at lock entry
coroutine 1:  resumes (still holds lock)  →  write counter=1  →  release lock
coroutine 2:  acquires lock  →  read counter=1  →  await  →  write counter=2  →  release
...
result: 100
```

The `await` inside the lock in the fixed version is safe because the lock is **still held** while the coroutine is suspended — no other coroutine can sneak into the critical section during that yield.

---

In [ ]:
import asyncio

counter = 0


async def increment():
    global counter
    await asyncio.sleep(0)  # yields control, another coroutine can run
    counter = counter + 1  # read & write, no await between


async def main():
    tasks = [increment() for _ in range(100)]
    await asyncio.gather(*tasks)
    print(counter)


# asyncio.run() fails in Jupyter because the kernel already runs an event loop.
# In Jupyter, top-level `await` is supported directly.
await main()

It works because the yield is now **before** the read-modify-write, not **inside** it:

```python
await asyncio.sleep(0)  # yield happens HERE — before any read
counter = counter + 1   # read AND write — no await between them
```

`counter = counter + 1` is a single Python statement with no `await` inside it. In asyncio, the **only** points where another coroutine can run are `await` expressions. So between the read of `counter` and the write back, no other coroutine can sneak in. Each coroutine reads the _current_ value, not a stale captured snapshot.

The sequence:

```shell
coroutine 1:  await (yields)
coroutine 2:  await (yields)
... all 100 yield first ...
coroutine 1:  counter = counter+1  → reads 0, writes 1  (no yield inside, uninterruptible)
coroutine 2:  counter = counter+1  → reads 1, writes 2  (same)
coroutine 3:  counter = counter+1  → reads 2, writes 3
...
result: 100
```

---

**⚠️ The critical contrast with threading:**

This exact same "fix" would **not** work with threads:

```python
# threading — this is NOT safe
def increment():
    time.sleep(0)
    counter = counter + 1  # OS can preempt between READ and WRITE
```

In threading, the OS can preempt a thread between _any two bytecodes_ — including between `LOAD_GLOBAL counter` and `STORE_GLOBAL counter` within a single statement. There is no concept of "no yield point here." You need a `Lock` regardless.

The rule that follows:

| Model         | Safe without a lock when...                                      |
| ------------- | ---------------------------------------------------------------- |
| **asyncio**   | No `await` between read and write                                |
| **threading** | Operation is a single atomic bytecode (almost nothing qualifies) |
